# 05 Run Report And Go/No-Go
Stage-Metriken aggregieren, Gate-Kriterien prüfen, Go/No-Go für nächsten Stage.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import sys

RUN_STAGE = "smoke"
RUN_ID = "replace_with_run_id_from_00"
def _find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src").exists() and (candidate / "configs").exists():
            return candidate.resolve()
    return start.resolve()

PROJECT_ROOT = _find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
from src.common.io_schema import read_parquet, validate_columns, MENTION_REQUIRED_COLUMNS
from src.common.run_report import evaluate_go_no_go, write_go_no_go_report

metrics_dir = PROJECT_ROOT / "artifacts/metrics" / RUN_ID
subset_dir = PROJECT_ROOT / "data/subsets/cache" / RUN_ID
cluster_dir = PROJECT_ROOT / "artifacts/clusters" / RUN_ID

stage_metrics = {}


In [ ]:
# Laufzeit-/Qualitätsindikatoren laden
train_manifest = {}
train_manifest_path = metrics_dir / "03_train_manifest.json"
if train_manifest_path.exists():
    with train_manifest_path.open("r", encoding="utf-8") as f:
        train_manifest = json.load(f)

lspo_mentions = read_parquet(subset_dir / f"lspo_mentions_{RUN_STAGE}.parquet")
ads_mentions = read_parquet(subset_dir / f"ads_mentions_{RUN_STAGE}.parquet")
clusters = read_parquet(cluster_dir / f"ads_clusters_{RUN_STAGE}.parquet") if (cluster_dir / f"ads_clusters_{RUN_STAGE}.parquet").exists() else pd.DataFrame(columns=["mention_id","block_key","author_uid"])

# schema checks
schema_valid = True
try:
    validate_columns(lspo_mentions, MENTION_REQUIRED_COLUMNS, "lspo_mentions")
    validate_columns(ads_mentions, MENTION_REQUIRED_COLUMNS, "ads_mentions")
except Exception:
    schema_valid = False

# determinism proxy: subset manifest exists and no duplicate mention_id
determinism_valid = bool((PROJECT_ROOT / "data/subsets/manifests" / f"{RUN_ID}_lspo_{RUN_STAGE}_manifest.parquet").exists())
uid_uniqueness_valid = True
if len(clusters):
    uid_uniqueness_valid = clusters.groupby("mention_id")["author_uid"].nunique().max() == 1

lspo_pairwise_f1 = None
if train_manifest.get("best_val_f1") is not None:
    lspo_pairwise_f1 = float(train_manifest["best_val_f1"])

stage_metrics.update({
    "run_id": RUN_ID,
    "stage": RUN_STAGE,
    "schema_valid": schema_valid,
    "determinism_valid": determinism_valid,
    "uid_uniqueness_valid": bool(uid_uniqueness_valid),
    "lspo_pairwise_f1": lspo_pairwise_f1,
    "counts": {
        "lspo_mentions": int(len(lspo_mentions)),
        "ads_mentions": int(len(ads_mentions)),
        "ads_clusters": int(len(clusters)),
    },
})

stage_metrics


In [ ]:
go = evaluate_go_no_go(stage_metrics)
report_path = metrics_dir / f"05_go_no_go_{RUN_STAGE}.json"
summary_path = metrics_dir / f"05_stage_metrics_{RUN_STAGE}.json"

with summary_path.open("w", encoding="utf-8") as f:
    json.dump(stage_metrics, f, indent=2)
write_go_no_go_report(go, report_path)

print("GO:" if go["go"] else "NO-GO:", go["go"])
display(pd.DataFrame(go["checks"]))
print("stage summary:", summary_path)
print("go/no-go:", report_path)
